# 02_Modeling_Evaluation: Huấn luyện và Đánh giá Mô hình Học máy

**Học phần:** Học máy (608807) – Lớp 69CS1  
**Đề tài 3:** Định giá Bất động sản Ames Housing  
**Người thực hiện:** Dương Hữu Hinh | MSSV: 0367869 (Phần việc: Pipeline & Thử nghiệm mô hình)

## 1. Đọc dữ liệu và chia train/test

Tập test (20%, `random_state=42`) đã được chia một lần và lưu Id ở `data/processed/test_ids.csv`.
Mọi phân tích và bước fit chỉ dùng **tập train**; tập test chỉ dùng để đánh giá cuối, tránh rò rỉ dữ liệu.

In [1]:
import sys
from pathlib import Path

# Thêm 03_SourceCode/ vào sys.path để import được thư mục src/
sys.path.append(str(Path.cwd().parent))

from src.data import RANDOM_STATE, load_split

X_train, X_test, y_train, y_test = load_split()
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (1168, 79) | Test: (292, 79)


## 2. Kiến trúc Pipeline

### Sơ đồ kiến trúc Pipeline 5 bước (đã chốt ngày 25/11):
```text
Dữ liệu thô (X_train)
      │
      ▼
1. MissingHandler (Học median LotFrontage theo khu & mode từ train fold)
      │
      ▼
2. OrdinalMapper (12 cột thứ bậc; giữ "passthrough" khi ordinal=False để tuning)
      │   (BsmtFinType1, BsmtFinType2, PoolQC không mã hóa thứ bậc)
      ▼
3. FeatureAdder (TotalSF, HouseAge, RemodAge, TotalBath, TotalPorchSF, flags; bỏ PoolQC)
      │
      ▼
4. ColumnTransformer (3 nhánh theo thứ tự num, cat, log):
   ├── num: SimpleImputer(median) ──► StandardScaler()
   ├── cat: SimpleImputer("None") ──► OneHotEncoder(min_frequency=0.01)
   └── log: 8 biến lệch phải ──► SimpleImputer(median) ──► log1p ──► StandardScaler()
      │
      ▼
5. Mô hình Hồi quy (Dummy, Ridge, Lasso, KNN, DecisionTree, RandomForest, HistGB)
   (Bọc trong TransformedTargetRegressor với func=np.log1p, inverse_func=np.expm1)
```

**Nguyên tắc chống rò rỉ dữ liệu (No Data Leakage):**
- Mọi bước học thông số từ dữ liệu (trung vị `LotFrontage` theo `Neighborhood`, mode của `Electrical`/`Bsmt*`, giá trị điền median, trung bình/độ lệch chuẩn của `StandardScaler`, danh mục của `OneHotEncoder`) đều nằm bên trong Pipeline, nên mỗi fold CV fit lại độc lập từ đầu.
- Hàm biến đổi `np.log1p` ở nhánh `log` không học tham số từ dữ liệu nhưng vẫn đặt trong `ColumnTransformer` để `predict.py` áp dụng đồng nhất khi suy luận dữ liệu mới.
- **Xử lý ngoại lai nằm ngoài Pipeline:** `remove_outliers()` (bỏ 3 căn `GrLivArea > 4000`) chỉ gọi trên tập train trước Pipeline vì transformer Scikit-learn không tự xóa bớt dòng dữ liệu.

In [2]:
from src.data import remove_outliers
from src.pipeline import build_pipeline, get_models

# Bỏ ngoại lai chỉ trên train (ngoài Pipeline vì transformer không xoá được dòng)
X_train, y_train = remove_outliers(X_train, y_train)

build_pipeline(get_models()["Ridge"])  # hiển thị sơ đồ Pipeline

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('missing',
                                                      MissingHandler()),
                                                     ('ordinal',
                                                      OrdinalMapper()),
                                                     ('fe', FeatureAdder()),
                                                     ('prep',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       Pipeline(steps=[('imputer',
                                                                                                        SimpleImputer(strategy='median')),
                                                                                                       ('scaler',
                                             

## 3. So sánh 7 mô hình bằng 5-Fold CV (chỉ trên train)

Đánh giá bằng `KFold(n_splits=5, shuffle=True, random_state=42)` trên tập train sạch (1.165 căn) qua 4 độ đo: RMSE (USD), MAE (USD), $R^2$ và RMSLE.

In [3]:
import pandas as pd
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "rmsle": "neg_root_mean_squared_log_error",
}

rows = {}
for name, model in get_models().items():
    res = cross_validate(
        build_pipeline(model), X_train, y_train, cv=cv, scoring=scoring
    )
    rows[name] = {
        "RMSE": -res["test_rmse"].mean(),
        "MAE": -res["test_mae"].mean(),
        "R2": res["test_r2"].mean(),
        "RMSLE": -res["test_rmsle"].mean(),
        "RMSLE_std": res["test_rmsle"].std(),
    }

bang_cv = pd.DataFrame(rows).T.sort_values("RMSLE")
bang_cv.round(4)

,RMSE,MAE,R2,RMSLE,RMSLE_std
Lasso,19955.0272,13481.5104,0.9299,0.1086,0.0157
Ridge,20122.7651,13551.6944,0.9288,0.1101,0.0133
HistGB,22940.8992,15142.7440,0.9069,0.1223,0.0140
RandomForest,25166.5647,16543.7412,0.8881,0.1330,0.0133
KNN,32445.6886,20602.3320,0.8141,0.1570,0.0089
DecisionTree,35799.1449,24535.9854,0.7740,0.1996,0.0102
Dummy,76722.8524,54266.3627,-0.0340,0.3887,0.0057


### Nhận xét kết quả 5-Fold Cross Validation:
1. **Mô hình tốt nhất:**
   - **Lasso** dẫn đầu với **RMSLE = 0.1086** ($\pm 0.0157$), $R^2 = 0.9299$, RMSE = \$19,955.03, MAE = \$13,481.51.
   - **Ridge** đứng thứ hai sát sao với **RMSLE = 0.1101** ($\pm 0.0133$), $R^2 = 0.9288$, RMSE = \$20,122.77, MAE = \$13,551.69.
2. **So sánh Tuyến tính (Lasso/Ridge) với Cây và Ensemble:**
   - Nhóm tuyến tính có điều chuẩn (Lasso 0.1086, Ridge 0.1101) vượt trội rõ rệt so với nhóm cây và ensemble (**HistGB**: 0.1223, **RandomForest**: 0.1330, **DecisionTree**: 0.1996).
   - Sau khi lấy `log1p` cho 8 biến diện tích/lệch phải cùng biến mục tiêu `SalePrice` và đưa `BsmtFinType1/2` sang one-hot, quan hệ giữa các đặc trưng và `log1p(SalePrice)` gần với tuyến tính hơn hẳn nên Lasso/Ridge khai thác hiệu quả; sai số CV ổn định giữa các fold (ý 4 bên dưới).
3. **Mức cải thiện so với Baseline (`DummyRegressor`):**
   - `Dummy` chỉ đạt RMSLE = 0.3887, $R^2 = -0.0340$, MAE = \$54,266.36.
   - Mô hình Lasso giảm sai số tuyệt đối trung bình MAE từ \$54,266.36 xuống còn \$13,481.51 (giảm 75.2% sai số), khẳng định Pipeline học được tín hiệu mạnh từ dữ liệu.
4. **Độ lệch chuẩn giữa các fold:**
   - Độ lệch chuẩn `RMSLE_std` của toàn bộ các mô hình đều nhỏ (từ 0.0057 đến 0.0157; Ridge 0.0133, Lasso 0.0157), cho thấy mô hình ổn định giữa các fold.

### Đối chứng phương thức mã hóa: `OrdinalMapper` (`ordinal=True`) vs. One-Hot (`ordinal=False`)

Thử nghiệm thay bước `ordinal` bằng `"passthrough"` (`ordinal=False`) để toàn bộ 12 cột thứ bậc còn lại đi vào One-Hot Encoding:

In [4]:
rows_no_ord = {}
for name, model in get_models().items():
    res = cross_validate(
        build_pipeline(model, ordinal=False), X_train, y_train, cv=cv, scoring=scoring
    )
    rows_no_ord[name] = {
        "RMSE": -res["test_rmse"].mean(),
        "MAE": -res["test_mae"].mean(),
        "R2": res["test_r2"].mean(),
        "RMSLE": -res["test_rmsle"].mean(),
        "RMSLE_std": res["test_rmsle"].std(),
    }

bang_cv_no_ord = pd.DataFrame(rows_no_ord).T.sort_values("RMSLE")
bang_cv_no_ord.round(4)

,RMSE,MAE,R2,RMSLE,RMSLE_std
Lasso,19539.1160,13085.3926,0.9328,0.1073,0.0155
Ridge,19622.9296,13133.1886,0.9322,0.1085,0.0134
HistGB,23177.3868,15191.0111,0.9049,0.1230,0.0153
RandomForest,25290.3313,16615.8833,0.8870,0.1335,0.0136
KNN,31965.8357,20121.5785,0.8195,0.1554,0.0089
DecisionTree,37106.4103,25084.3352,0.7578,0.1959,0.0139
Dummy,76722.8524,54266.3627,-0.0340,0.3887,0.0057


**Nhận xét đối chứng Ordinal vs. One-Hot:**
- Khi dùng One-Hot cho toàn bộ các cột chất lượng (`ordinal=False`), **Lasso** đạt RMSLE = 0.1073 ($R^2 = 0.9328$), **Ridge** đạt 0.1085 ($R^2 = 0.9322$), nhỉnh hơn nhẹ so với bản mặc định (0.1086 và 0.1101); ngược lại **HistGB** tăng từ 0.1223 lên 0.1230.
- Do kết quả trái chiều giữa nhóm tuyến tính và mô hình cây (và còn phụ thuộc hệ số điều chuẩn `alpha` chưa tuning), nhóm giữ **`OrdinalMapper` làm mặc định** để bảo toàn thứ bậc tự nhiên và đưa lựa chọn `regressor__ordinal` ∈ `{OrdinalMapper(), "passthrough"}` vào bước tuning ở Mục 4.

## 4. Tinh chỉnh siêu tham số (Hyperparameter Tuning)

> **Phân công nhóm:** Mục này do bạn **Ngô Quang Anh** tiếp tục triển khai (`RandomizedSearchCV` / `GridSearchCV` với `regressor__ordinal` và `regressor__model__*`) sau khi merge nhánh `hinh`.

## 5. Đánh giá cuối cùng trên tập kiểm tra độc lập (Test Set)

> **Phân công nhóm:** Mục này do bạn **Ngô Quang Anh** thực hiện đánh giá một lần duy nhất cho mô hình tối ưu cuối cùng trên `X_test, y_test` (sử dụng hàm `src.plots.plot_model_evaluation` để xuất biểu đồ 4 góc).